# USAspending bulk awards — demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/usaspending/blob/main/demo.ipynb)

Query the [`abigailhaddad/usaspending-bulk-awards`](https://huggingface.co/datasets/abigailhaddad/usaspending-bulk-awards) dataset directly with DuckDB — no download, no auth. The data is prime federal **contracts** and **assistance** (grants/loans/etc.), FY2007–present, all agencies, as typed Parquet partitioned by `fiscal_year` and `agency`.

DuckDB reads straight from HuggingFace over `hf://` and uses **partition pruning**, so filtering on `fiscal_year`/`agency` only reads the relevant files.

In [ ]:
!pip install -q duckdb pandas plotly

In [ ]:
import duckdb

REPO = "abigailhaddad/usaspending-bulk-awards"
BASE = f"hf://datasets/{REPO}"
CONTRACTS = f"read_parquet('{BASE}/contracts/**/*.parquet', hive_partitioning=true)"
ASSISTANCE = f"read_parquet('{BASE}/assistance/**/*.parquet', hive_partitioning=true)"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
# Public dataset: no token needed. (If the dataset is still private, uncomment:)
# import os; con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

def q(sql):
    return con.execute(sql).df()

## What's available
Coverage by fiscal year (rows + total obligations).

In [ ]:
q(f"""
  SELECT fiscal_year,
         count(*) AS transactions,
         round(sum(federal_action_obligation)/1e9, 2) AS obligations_billions
  FROM {CONTRACTS}
  GROUP BY 1 ORDER BY 1
""")

## Contract spend over time

In [ ]:
import plotly.express as px

ot = q(f"""
  SELECT fiscal_year, sum(federal_action_obligation)/1e9 AS obligations_billions
  FROM {CONTRACTS}
  GROUP BY 1 ORDER BY 1
""")
px.bar(ot, x='fiscal_year', y='obligations_billions',
       title='Federal contract obligations by fiscal year ($B)')

## Top contract recipients in a given year
Partition pruning: filtering `fiscal_year` reads only that year's files.

In [ ]:
q(f"""
  SELECT recipient_name,
         round(sum(federal_action_obligation)/1e6, 1) AS obligations_millions,
         count(*) AS transactions
  FROM {CONTRACTS}
  WHERE fiscal_year = '2007'
  GROUP BY 1 ORDER BY 2 DESC
  LIMIT 15
""")

## Top agencies by contract spend

In [ ]:
q(f"""
  SELECT awarding_agency_name,
         round(sum(federal_action_obligation)/1e9, 2) AS obligations_billions
  FROM {CONTRACTS}
  WHERE fiscal_year = '2007'
  GROUP BY 1 ORDER BY 2 DESC
  LIMIT 15
""")

## Contracts vs. assistance
The two products have different schemas but share `federal_action_obligation` and `fiscal_year`.

In [ ]:
q(f"""
  SELECT 'contracts' AS kind, fiscal_year,
         round(sum(federal_action_obligation)/1e9, 2) AS obligations_billions
  FROM {CONTRACTS} WHERE fiscal_year = '2007' GROUP BY 1,2
  UNION ALL
  SELECT 'assistance', fiscal_year,
         round(sum(federal_action_obligation)/1e9, 2)
  FROM {ASSISTANCE} WHERE fiscal_year = '2007' GROUP BY 1,2
""")

## Reference tables
Small dimension tables ship alongside the data. The **data dictionary** documents every column in both products.

In [ ]:
q(f"""
  SELECT "Element", "Definition", "Award File"
  FROM read_parquet('{BASE}/reference/data_dictionary.parquet')
  WHERE "Award Element" = 'federal_action_obligation'
""")

## Export a filtered slice to CSV

In [ ]:
df = q(f"""
  SELECT recipient_name, awarding_agency_name, action_date,
         federal_action_obligation, naics_description,
         product_or_service_code_description
  FROM {CONTRACTS}
  WHERE fiscal_year = '2007' AND federal_action_obligation > 1e8
  ORDER BY federal_action_obligation DESC
""")
df.to_csv('usaspending_slice.csv', index=False)
print(f'{len(df):,} rows written to usaspending_slice.csv')
df.head()